# Chapter 7 &mdash; Theorem: $L$ is Regular iff Some NFA Recognizes It

**Concept 9 of the Chapter 7 decomposition:** *Theorem: $L$ is Regular iff Some NFA Recognizes It*

Both directions in two lines &mdash; every DFA is an NFA, and subset construction converts back.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Regular-Iff-NFA/Concept-Regular-Iff-NFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


> **Theorem.** $L$ is regular $\iff$ some NFA recognizes $L$.

**($\Rightarrow$)** A DFA **is** an NFA: take $Q_0=\{q_0\}$ and read each
$\delta(q,a)=q'$ as $\{q'\}$. Nothing to prove.

**($\Leftarrow$)** Given an NFA, the subset construction yields a DFA for the same
language. Concept 8 is the proof.

Two lines each &mdash; and the payoff is large: from here on you may design with
whichever model is convenient and convert when you need the other. Chapters 8&ndash;10 lean
on this constantly.

## 2. Definitions

### Direction 1: a DFA, viewed as an NFA

In [ ]:
D = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')

def dfa_as_nfa(D):
    Dl = {(q, a): {t} for (q, a), t in D["Delta"].items()}
    return mk_nfa(D["Q"], D["Sigma"], Dl, {D["q0"]}, D["F"])

### Direction 2: an NFA, converted by subset construction

In [ ]:
N = md2mc('''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 0 | 1 -> F
''')

## 3. Tests

Direction 1: the reading is trivially faithful.

In [ ]:
As_nfa = dfa_as_nfa(D)
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
assert all(accepts_nfa(As_nfa, s) == accepts_dfa(D, s) for s in strs)
print("DFA-as-NFA accepts exactly what the DFA does, on all %d strings" % len(strs))
print("|Q0| =", len(As_nfa["Q0"]), " -- a singleton, which is what makes it deterministic")

Direction 2: `nfa2dfa` closes the loop.

In [ ]:
Back = nfa2dfa(N)
assert all(accepts_dfa(Back, s) == accepts_nfa(N, s) for s in strs)
print("subset construction preserves the language on all %d strings" % len(strs))

Round trip: DFA &rarr; NFA &rarr; DFA returns an isomorphic minimal machine.

In [ ]:
round_trip = min_dfa(nfa2dfa(dfa_as_nfa(D)))
print("original minimal %d states, round-tripped %d states"
      % (len(min_dfa(D)["Q"]), len(round_trip["Q"])))
assert iso_dfa(min_dfa(D), round_trip)
print("isomorphic? ", iso_dfa(min_dfa(D), round_trip))

So the two models describe **exactly** the same class of languages.

In [ ]:
print("regular languages  ==  DFA languages  ==  NFA languages")
print("\nChapters 8-10 add regular expressions to that chain.")

## 4. Animation

The NFA and its determinization recognise one language; here is the DFA.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(nfa2dfa(N)), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Write out direction ($\Rightarrow$) as a formal proof. How long is it really?
2. Does the round trip ever *grow* the minimal DFA? Why not?
3. What would change if NFA acceptance were "**all** copies land in $F$"?

In [ ]:
# Your work for the exercises above.